# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadbinijaz17/flyrankAI_Intern_ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Bootstrap: run identically in Colab and locally (copied from the starter notebooks).
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/muhammadbinijaz17/flyrankAI_Intern_ML"
REPO_DIR = "flyrankAI_Intern_ML"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # walk up to the repo root (folder containing data/raw)
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Working dir:", os.getcwd())
print("Starter data found. You're ready.")

Working dir: D:\FlyrankAI\flyrankAI_Intern_ML
Starter data found. You're ready.


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

To construct an honest, leak-free feature vector from `content_refresh_anonymized.csv`:
1. **Heavy-tail transformations:** Log-transform raw counts using $\log(1 + x)$ (`log_impressions_90d`, `log_clicks_90d`, `log_sessions_90d`, `log_ai_sessions_90d`) to prevent extreme power-law outliers from dominating linear and tree splits.
2. **Explicit sparsity flags:** Instead of blindly filling missing values with zero (which injects a silent categorical signal because missingness is systematic by `content_type`), we add explicit indicator flags: `has_clicks`, `has_ai_sessions`, `has_search_volume`, `has_word_count`, and `measurable_opportunity` (defined as $\ge 100$ impressions and $> 0$ sessions).
3. **Rate scaling and bounds:** Rates (`ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`) are preserved as $\times 100$ percentages, with `avg_position = 0` acknowledged as an unranked indicator.
4. **Categorical encoding:** Discretized tiers and categorical fields (`competition_level`, `content_type`, `main_intent`, `age_tier`, `freshness_tier`, `word_count_tier`, `impression_tier`, `position_tier`) have missing values imputed as `"unknown"` and are one-hot encoded.
5. **Clean output:** Produces a final feature matrix $X_{\text{clean}}$ of 30,000 rows $\times$ 52 columns.

In [3]:
import pandas as pd
import numpy as np

# Load raw dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 1. Define target
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# 2. Engineered feature transformations
df["log_impressions_90d"] = np.log1p(df["impressions_90d"].clip(lower=0))
df["log_clicks_90d"] = np.log1p(df["clicks_90d"].clip(lower=0))
df["log_sessions_90d"] = np.log1p(df["sessions_90d"].clip(lower=0))
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"].clip(lower=0))

# 3. Missingness and activity indicator flags
df["has_clicks"] = (df["clicks_90d"] > 0).astype(int)
df["has_ai_sessions"] = (df["ai_sessions_90d"] > 0).astype(int)
df["has_search_volume"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["measurable_opportunity"] = ((df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)).astype(int)

# 4. Numerical feature matrix
numeric_cols = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "has_clicks", "has_ai_sessions", "has_search_volume", "has_word_count", "measurable_opportunity"
]
X_num = df[numeric_cols].fillna(0)

# 5. Categorical one-hot matrix
cat_cols = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier"
]
X_cat = pd.get_dummies(df[cat_cols].fillna("unknown"), drop_first=True)

# 6. Final clean feature matrix
X_clean = pd.concat([X_num, X_cat], axis=1)
y = df["is_declining_label"].values
groups = df["client_id"].values

print(f"Dataset rows: {len(df):,}")
print(f"Engineered feature matrix shape: {X_clean.shape}")
print(f"Target distribution (base rate): {y.mean():.2%} positive ({y.sum():,} rows)")
print(f"Number of client groups: {len(np.unique(groups))}")

Dataset rows: 30,000
Engineered feature matrix shape: (30000, 52)
Target distribution (base rate): 54.21% positive (16,262 rows)
Number of client groups: 32


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature Group | Features | Meaning & Measurement | Missingness Strategy | Temporal Availability (Knowable When?) |
|---|---|---|---|---|
| **Keyword Demand** | `search_volume`, `competition`, `competition_level`, `cpc`, `has_search_volume` | Third-party search market demand estimates for target keyword | Missing in 100% of feedly articles (2,468 rows). Imputed to 0 + explicit indicator `has_search_volume=0` | **Knowable BEFORE:** Static keyword landscape data determined prior to review moment |
| **Content Properties** | `word_count`, `char_count`, `content_age_days`, `days_since_last_update`, `has_word_count`, `*_tier` | Article physical length and lifecycle recency | `word_count` missing in 28.3% of keyword articles (7,699 rows). Imputed to 0 + indicator `has_word_count=0` | **Knowable BEFORE:** Fixed page metadata at snapshot moment |
| **90-Day Activity & Logs** | `log_impressions_90d`, `log_clicks_90d`, `log_sessions_90d`, `log_ai_sessions_90d`, `has_clicks`, `has_ai_sessions` | Total exposure and engagement volume over trailing 90 days | Zero-filled for unobserved channels; log1p-transformed for heavy tails | **Knowable BEFORE:** Trailing cumulative history ending at review cutoff |
| **Rates & Efficiency** | `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct` | Relative engagement rates (all $\times 100$). `avg_position=0` means unranked | Zero-filled where traffic is absent | **Knowable BEFORE:** Observed rate over the trailing window |
| **Historical Baseline** | `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` | Traffic in days 31–60 back from export (the baseline period) | Zero-filled if no traffic recorded | **Knowable BEFORE:** Strictly precedes the last-30-day outcome window |

In [5]:
# Section 2 check: Feature dictionary table and timing validation
feature_metadata = []
for col in numeric_cols:
    raw_missing = df[col].isna().mean() if col in df.columns else 0.0
    feature_metadata.append({
        "feature": col,
        "type": "numeric",
        "missing_pct": f"{raw_missing:.1%}",
        "fill": "0 + has_ flag" if "has_" in col or col in ["search_volume", "word_count"] else "0",
        "available_before_prediction": "YES (Historical / Metadata)"
    })

meta_df = pd.DataFrame(feature_metadata)
print(f"Verified {len(meta_df)} numerical & engineered features:")
print(meta_df.head(10).to_string(index=False))
print(f"... and {X_cat.shape[1]} one-hot categorical indicator columns.")

Verified 26 numerical & engineered features:
              feature    type missing_pct          fill available_before_prediction
        search_volume numeric        8.2% 0 + has_ flag YES (Historical / Metadata)
          competition numeric        8.2%             0 YES (Historical / Metadata)
                  cpc numeric        8.2%             0 YES (Historical / Metadata)
           word_count numeric       25.7% 0 + has_ flag YES (Historical / Metadata)
           char_count numeric       25.7%             0 YES (Historical / Metadata)
  log_impressions_90d numeric        0.0%             0 YES (Historical / Metadata)
       log_clicks_90d numeric        0.0%             0 YES (Historical / Metadata)
     log_sessions_90d numeric        0.0%             0 YES (Historical / Metadata)
  log_ai_sessions_90d numeric        0.0%             0 YES (Historical / Metadata)
days_with_impressions numeric        0.0%             0 YES (Historical / Metadata)
... and 26 one-hot categorical 

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

We rigorously attack our own model using the taxonomy from `skills/hunting-leakage-and-validating/SKILL.md`:

### Attack 1: Label-Derived Features (`trend_pct` and `trend_direction`)
- **Vulnerability:** `is_declining_label` is generated by the rule `trend_direction == 'down'` (which is computed from `trend_pct < -20.0`).
- **The Confession Test:** We train a model **WITH** `trend_pct` and **WITHOUT** `trend_pct`. Including `trend_pct` produces an in-sample ROC-AUC of **1.0000** (the tree memorizes the exact -20% split threshold). Removing it reveals the honest, non-tautological performance (**0.7022** GroupKFold AUC).

### Attack 2: Future / Outcome Window Overlap
- **Vulnerability:** `impressions_last_30d`, `clicks_last_30d`, and `sessions_last_30d` measure activity in days 1–30 back — the exact period determining whether the page dropped. Feeding last-30-day volume directly into the model allows it to "cheat" by observing the outcome.
- **Defense:** All `*_last_30d` columns are strictly excluded. Only `*_prev_30d` (days 31–60 back) and trailing totals are retained.

### Attack 3: Group Memorization vs Honest Out-of-Domain Generalization
- **Vulnerability:** Pages from the same client share domain authority, CMS structures, and topic niches. A standard random train/test split allows the model to memorize client-specific signals.
- **Defense:** We evaluate with **5-Fold GroupKFold by `client_id`** (held-out client evaluation) alongside random 5-Fold CV:
  - **Random 5-Fold CV AUC:** **0.7775** (optimistic due to domain memorization).
  - **Honest GroupKFold 5-Fold CV AUC:** **0.7022** (honest generalization to completely unseen client websites).
  - The **~7.5-point generalization gap** is an empirical finding proving why grouped splitting is mandatory.

In [7]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, GroupKFold, cross_val_score
from sklearn.metrics import roc_auc_score

# 1. Test Leakage Attack (With vs Without trend_pct)
X_leaky = pd.concat([X_clean, df[["trend_pct"]].fillna(0)], axis=1)
tree_leaky = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y)
auc_leaky = roc_auc_score(y, tree_leaky.predict_proba(X_leaky)[:, 1])

print("=== ATTACK 1: LEAKY MODEL WITH trend_pct ===")
print(f"In-sample ROC-AUC: {auc_leaky:.4f} (Near-perfect 1.00 = Confession of Leakage!)")
print("Decision Tree Splits (note the tautological split on trend_pct):")
print(export_text(tree_leaky, feature_names=list(X_leaky.columns), max_depth=2))

# 2. Honest Model Evaluation (Random CV vs GroupKFold CV)
model_honest = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)

cv_random = KFold(n_splits=5, shuffle=True, random_state=42)
scores_random = cross_val_score(model_honest, X_clean, y, cv=cv_random, scoring="roc_auc")

cv_group = GroupKFold(n_splits=5)
scores_group = cross_val_score(model_honest, X_clean, y, groups=groups, cv=cv_group, scoring="roc_auc")

print("=== ATTACK 2 & 3: HONEST EVALUATION & GENERALIZATION GAP ===")
print(f"Base Rate (Declining Class): {y.mean():.2%}")
print(f"Random 5-Fold CV AUC:    {scores_random.mean():.4f} +/- {scores_random.std():.4f}")
print(f"GroupKFold 5-Fold CV AUC: {scores_group.mean():.4f} +/- {scores_group.std():.4f}")
print(f"Generalization Gap:       {(scores_random.mean() - scores_group.mean()):.4f} (points lost when predicting unseen clients)")

=== ATTACK 1: LEAKY MODEL WITH trend_pct ===
In-sample ROC-AUC: 1.0000 (Near-perfect 1.00 = Confession of Leakage!)
Decision Tree Splits (note the tautological split on trend_pct):
|--- trend_pct <= -20.05
|   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- impressions_prev_30d <= 5700.00
|   |   |   |--- class: 0
|   |   |--- impressions_prev_30d >  5700.00
|   |   |   |--- class: 1
|   |--- trend_pct >  -19.95
|   |   |--- class: 0

=== ATTACK 2 & 3: HONEST EVALUATION & GENERALIZATION GAP ===
Base Rate (Declining Class): 54.21%
Random 5-Fold CV AUC:    0.7775 +/- 0.0052
GroupKFold 5-Fold CV AUC: 0.7022 +/- 0.0562
Generalization Gap:       0.0753 (points lost when predicting unseen clients)


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded Field | Role / Data Type | One-Line Rationale for Exclusion |
|---|---|---|
| `trend_direction` | String / Categorical | **Target Leakage:** Directly defines the outcome label (`is_declining_label = (trend_direction == 'down')`). |
| `trend_pct` | Float / Percentage | **Formula Leakage:** The mathematical continuous input used to compute `trend_direction` ($< -20\%$). |
| `impressions_last_30d` | Integer / Count | **Outcome Window Leakage:** Measures search volume during the evaluated outcome window (future information). |
| `clicks_last_30d` | Integer / Count | **Outcome Window Leakage:** Measures click volume during the evaluated outcome window. |
| `sessions_last_30d` | Integer / Count | **Outcome Window Leakage:** Measures site traffic during the evaluated outcome window. |
| `content_id` | String / Pseudonym | **Context Key:** Unique row hash; memorizes specific pages without generalizable predictive value. |
| `client_id` | String / Pseudonym | **Grouping Variable:** Client organization identifier; reserved strictly for GroupKFold validation splitting. |
| `provider_used` | String / Categorical | **Tooling Metadata:** 71.5% missing operational tag for LLM provider; uninformative for search demand. |
| `model_used` | String / Categorical | **Tooling Metadata:** 19.1% missing operational tag for LLM model; uninformative for organic search trends. |

In [9]:
# Section 4 check: Verify zero excluded columns exist in X_clean & check feature importances
excluded_cols = [
    "trend_direction", "trend_pct", "impressions_last_30d", "clicks_last_30d",
    "sessions_last_30d", "content_id", "client_id", "provider_used", "model_used"
]

for col in excluded_cols:
    assert col not in X_clean.columns, f"CRITICAL ERROR: Excluded column {col} was found in X_clean!"

print(f"Verification passed: None of the {len(excluded_cols)} excluded columns exist in X_clean.")

# Fit honest model and check top feature importances
model_honest.fit(X_clean, y)
importances = pd.Series(model_honest.feature_importances_, index=X_clean.columns).sort_values(ascending=False)

print("\nTop 10 Feature Importances in Honest Model:")
for rank, (feat, imp) in enumerate(importances.head(10).items(), 1):
    print(f"{rank:2d}. {feat:30s} {imp:.4f} ({imp:.1%})")

# Ensure no single feature dominates with > 50% importance
assert importances.iloc[0] < 0.50, f"Suspicious dominance: {importances.index[0]} holds {importances.iloc[0]:.2%} importance!"

Verification passed: None of the 9 excluded columns exist in X_clean.

Top 10 Feature Importances in Honest Model:
 1. impressions_prev_30d           0.3069 (30.7%)
 2. days_with_impressions          0.0951 (9.5%)
 3. log_impressions_90d            0.0735 (7.3%)
 4. avg_position                   0.0702 (7.0%)
 5. content_age_days               0.0531 (5.3%)
 6. char_count                     0.0413 (4.1%)
 7. measurable_opportunity         0.0388 (3.9%)
 8. position_tier_top_3            0.0332 (3.3%)
 9. age_tier_365+                  0.0260 (2.6%)
10. log_clicks_90d                 0.0258 (2.6%)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.